# Challenge 3: Developing Multi-Agent Systems
A root agent that coordinates a weather agent and a search agent as sub-agents.

In [11]:
!pip install google-adk requests

## Setup imports and environment

In [12]:
import requests
import os
import logging
import time
from typing import Optional, List, Dict
from google.adk import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from google.adk.tools import google_search
from google.adk.tools import AgentTool

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = "qwiklabs-gcp-01-fab96dd167ae"
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

GOOGLE_MAPS_API_KEY = "YOUR_API_KEY_GOES_HERE"

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logging.getLogger("google_genai.types").setLevel(logging.ERROR)

## Tool functions for the weather agent

In [13]:
def get_lat_lon(location: str) -> Dict[str, float]:
    """Convert a place name to latitude and longitude using Google Maps Geocoding API.

    Args:
        location: A place name or address (e.g., "Denver, CO").

    Returns:
        A dictionary with 'lat' and 'lng' keys, or an 'error' key if not found.
    """
    url = "https://maps.googleapis.com/maps/api/geocode/json"
    params = {"address": location, "key": GOOGLE_MAPS_API_KEY}
    resp = requests.get(url, params=params)
    data = resp.json()
    if data["status"] == "OK":
        loc = data["results"][0]["geometry"]["location"]
        return {"lat": loc["lat"], "lng": loc["lng"]}
    return {"error": f"Geocoding failed: {data['status']}"}

In [14]:
def get_extended_weather_forecast(lat: float, lon: float) -> Optional[List[Dict[str, str]]]:
    """Retrieve the extended weather forecast from the National Weather Service API.

    Args:
        lat: Latitude of the location.
        lon: Longitude of the location.

    Returns:
        A list of forecast periods with 'name', 'temperature', and 'forecast' keys,
        or None if the request fails.
    """
    headers = {"User-Agent": "weather-agent-app"}
    points_url = f"https://api.weather.gov/points/{lat},{lon}"
    resp = requests.get(points_url, headers=headers)
    if resp.status_code != 200:
        return None
    forecast_url = resp.json()["properties"]["forecast"]

    resp = requests.get(forecast_url, headers=headers)
    if resp.status_code != 200:
        return None
    periods = resp.json()["properties"]["periods"]
    return [
        {
            "name": p["name"],
            "temperature": f"{p['temperature']}\u00b0{p['temperatureUnit']}",
            "forecast": p["detailedForecast"],
        }
        for p in periods
    ]

## Sub-agent 1: Weather Agent
Handles weather-related queries using NWS API and geocoding.

In [15]:
weather_agent = Agent(
    name="weather_agent",
    model="gemini-2.0-flash",
    description="Handles weather forecast requests for US locations. Use this agent when the user asks about weather, forecasts, or temperature.",
    instruction="""You are a weather specialist. When asked about weather:
1. Use get_lat_lon to convert the location to coordinates.
2. Use get_extended_weather_forecast to retrieve the forecast.
3. Summarize the forecast in a friendly, readable way.
Only handle weather-related questions. For other topics, say you can only help with weather.""",
    tools=[get_lat_lon, get_extended_weather_forecast],
)

## Sub-agent 2: Search Agent
Handles general knowledge queries using Google Search.

In [16]:
search_agent = Agent(
    name="search_agent",
    model="gemini-2.0-flash",
    description="Handles general knowledge and current events queries using Google Search. Use this agent for non-weather questions.",
    instruction="""You are a search specialist. Use Google Search to find up-to-date information
and answer the user's question clearly and concisely.""",
    tools=[google_search],
)

## Root Agent
Coordinates between the weather and search sub-agents based on the user's request.

In [17]:
root_agent = Agent(
    name="root_agent",
    model="gemini-2.0-flash",
    description="A coordinating agent that delegates tasks to specialized sub-agents.",
    instruction="""You are a helpful assistant that coordinates specialized agents.
When the user asks about weather, forecasts, or temperature, delegate to the weather_agent.
When the user asks about general knowledge, news, or current events, use the search_agent tool.
Always provide a helpful response based on what the sub-agents return.""",
    tools=[AgentTool(agent=search_agent)],
    sub_agents=[weather_agent],
)

## Helper to run the agent and display all events
Outputs events to demonstrate which sub-agents are being used.

In [18]:
async def run_agent_verbose(agent, user_message: str) -> str:
    """Run the agent, print all events to show sub-agent delegation, and return the final response."""
    session_service = InMemorySessionService()
    session = await session_service.create_session(
        app_name=agent.name, user_id="test_user"
    )
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)

    content = types.Content(
        role="user", parts=[types.Part(text=user_message)]
    )
    final_response = ""
    async for event in runner.run_async(
        user_id="test_user", session_id=session.id, new_message=content
    ):
        # Print event details to show sub-agent usage
        if event.author:
            print(f"  [Event] author={event.author}, is_final={event.is_final_response()}")
        if event.is_final_response() and event.content and event.content.parts:
            final_response = event.content.parts[0].text
    return final_response

## Test: Queries that exercise both sub-agents
- Weather queries should be delegated to weather_agent
- General queries should be delegated to search_agent

In [19]:
test_queries = [
    "What's the weather in Springfield, Illinois?",             # -> weather_agent
    "Who won the Super Bowl in 2025?",                     # -> search_agent
    "What's the extended forecast for Miami, Florida?",             # -> weather_agent
    "What are the latest headlines about AI?",             # -> search_agent
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"Query: {query}")
    print(f"{'='*60}")
    result = await run_agent_verbose(root_agent, query)
    print(f"\nResponse: {result}")
    time.sleep(5)


Query: What's the weather in Springfield, Illinois?
  [Event] author=root_agent, is_final=False
  [Event] author=root_agent, is_final=False
  [Event] author=weather_agent, is_final=False
  [Event] author=weather_agent, is_final=False
  [Event] author=weather_agent, is_final=False
  [Event] author=weather_agent, is_final=False
  [Event] author=weather_agent, is_final=True

Response: The weather in Springfield, Illinois: This afternoon will be mostly cloudy with a high near 61°F. Tonight, there will be widespread fog before midnight, then a chance of rain and patchy fog with a slight chance of thunderstorms. The low will be around 52°F. Friday has a chance of thunderstorms and rain, with a high near 78°F. There is a high chance of showers and thunderstorms on Friday night, with a low around 57°F. Saturday will be partly cloudy with a high near 63°F.

Query: Who won the Super Bowl in 2025?
  [Event] author=root_agent, is_final=False
  [Event] author=root_agent, is_final=False
  [Event] a